In [1]:
%%capture
import os
!pip install unsloth gradio bitsandbytes accelerate
!pip install --no-deps unsloth_zoo
!pip install transformers==4.56.2

In [2]:
import gradio as gr
import torch
import json
import random
from PIL import Image
from unsloth import FastVisionModel

# --- 1. CẤU HÌNH ĐƯỜNG DẪN ---
MODEL_NAME = "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit"
ADAPTER_PATH = "/kaggle/input/notebooks/spixalo/dl-endterm-b2-vnts/checkpoints_2b/checkpoint-750/"

# Thêm đường dẫn cho tập Test
IMAGE_TEST_DIR = "/kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive/images" 
TEST_JSON_PATH = "/kaggle/input/datasets/spixalo/trafic/traffic-signs-json/test.json"

print("Đang triệu hồi các 'pháp sư' AI và nạp đạn (dữ liệu)...")

# --- 2. LOAD DỮ LIỆU TEST ---
test_data_pool = []
try:
    with open(TEST_JSON_PATH, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
        for item in raw_data:
            test_data_pool.append({
                "img_path": f"{IMAGE_TEST_DIR}/{item['image_name']}",
                "question": item["question"],
                "answer": item["answer"][0]
            })
    print(f"Đã nạp {len(test_data_pool)} ảnh từ tập Test để quay xổ số!")
except Exception as e:
    print(f"Lỗi nạp dữ liệu test (nhưng không sao, dùng ảnh tự up vẫn được): {e}")

# --- 3. LOAD MODEL B1 & B2 ---
model_b1, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME, load_in_4bit=True, 
)
FastVisionModel.for_inference(model_b1)

model_b2, _ = FastVisionModel.from_pretrained(
    MODEL_NAME, load_in_4bit=True, 
)
model_b2.load_adapter(ADAPTER_PATH)
FastVisionModel.for_inference(model_b2)

# --- 4. CÁC HÀM XỬ LÝ ---
def predict_vqa(image, question):
    if image is None or not question:
        return "Nè, nạp ảnh với câu hỏi vào mới chấm điểm được chứ!", ""

    prompt = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": question}]}]
    inputs = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out_b1 = model_b1.generate(**inputs, max_new_tokens=20, use_cache=True)
        out_b2 = model_b2.generate(**inputs, max_new_tokens=20, use_cache=True)
        
    ans_b1 = tokenizer.decode(out_b1[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    ans_b2 = tokenizer.decode(out_b2[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    return ans_b1.strip(), ans_b2.strip()

def get_random_test_sample():
    if not test_data_pool:
        return None, "Không tìm thấy dữ liệu test, kiểm tra lại đường dẫn nhé!", "N/A"
    
    sample = random.choice(test_data_pool)
    try:
        img = Image.open(sample["img_path"]).convert("RGB")
        # Trả về: Ảnh, Câu hỏi, và in luôn Đáp án đúng ra giao diện cho giảng viên xem
        return img, sample["question"], f"Đáp án đúng (Ground Truth): {sample['answer']}"
    except Exception as e:
        return None, f"Lỗi mở ảnh: {e}", "N/A"

# --- 5. GIAO DIỆN GRADIO ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚦 Hệ thống VQA Biển báo Giao thông Việt Nam")
    gr.Markdown("So sánh trực tiếp giữa **B1 (Zero-shot)** và **B2 (Fine-tuned)**. Bạn có thể tự tải ảnh lên hoặc bốc ngẫu nhiên từ tập Test!")
    
    with gr.Row():
        with gr.Column(scale=1):
            input_img = gr.Image(type="pil", label="Ảnh biển báo")
            
            # Nút bấm ma thuật nằm ở đây
            btn_random = gr.Button("🎲 Lấy ảnh ngẫu nhiên từ tập Test", variant="secondary")
            
            input_txt = gr.Textbox(label="Câu hỏi của bạn", placeholder="Ví dụ: Đây là biển báo gì?")
            ground_truth_txt = gr.Textbox(label="Gợi ý đáp án từ Dataset", interactive=False, visible=True)
            
            btn_predict = gr.Button("🔍 Phân tích ngay", variant="primary")
            
        with gr.Column(scale=1):
            out_b1 = gr.Textbox(label="🤖 Thí sinh B1 (Zero-shot)", interactive=False)
            out_b2 = gr.Textbox(label="🏆 Thí sinh B2 (Fine-tuned)", interactive=False)
            
    # Nối dây điện cho các nút
    btn_random.click(
        fn=get_random_test_sample, 
        inputs=[], 
        outputs=[input_img, input_txt, ground_truth_txt]
    )
    
    btn_predict.click(
        fn=predict_vqa, 
        inputs=[input_img, input_txt], 
        outputs=[out_b1, out_b2]
    )

demo.launch(share=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-08 02:07:48.294588: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778206068.515494      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778206068.582137      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778206069.113723      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778206069.113793      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778206069.113799      22 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Đang triệu hồi các 'pháp sư' AI và nạp đạn (dữ liệu)...
Đã nạp 260 ảnh từ tập Test để quay xổ số!
==((====))==  Unsloth 2026.5.2: Fast Qwen2_Vl patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

==((====))==  Unsloth 2026.5.2: Fast Qwen2_Vl patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/tmp/ipykernel_22/3965213907.py:75: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://47c4b8f54986687fff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
